# Assignment 7 - Transfer Learning
**Topic:** 2026-27 - TY E2 - DL Lab
**Description:** Implement Transfer Learning using pre-trained Alexnet, VGG-16, RESNET50, EfficientNetB0 models for image classification and compare their performance.

## 1. Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D, Input
import matplotlib.pyplot as plt
import numpy as np

## 2. Load and Preprocess CIFAR-10 Dataset

In [2]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train, y_train = x_train[:3000], y_train[:3000]
x_test, y_test = x_test[:1000], y_test[:1000]

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1593s 9us/step


c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


x_train shape: (3000, 32, 32, 3), y_train shape: (3000, 10)


## 3. Function to Build and Train Models

In [3]:
def train_transfer_model(base_model_func, preprocess_func, name):
    print(f"\n--- Training {name} ---")
    x_train_prep = preprocess_func(x_train.astype('float32'))
    x_test_prep = preprocess_func(x_test.astype('float32'))
    
    base_model = base_model_func(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
    base_model.trainable = False
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions = Dense(10, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    history = model.fit(x_train_prep, y_train, epochs=2, batch_size=64, validation_data=(x_test_prep, y_test), verbose=1)
    loss, accuracy = model.evaluate(x_test_prep, y_test, verbose=0)
    return accuracy, history

## 4. Train Models
*Note: AlexNet is not natively available in tf.keras.applications, so we compare VGG16, ResNet50, and EfficientNetB0.*

In [ ]:
results = {}

acc_vgg, hist_vgg = train_transfer_model(VGG16, tf.keras.applications.vgg16.preprocess_input, 'VGG16')
results['VGG16'] = acc_vgg

acc_resnet, hist_resnet = train_transfer_model(ResNet50, tf.keras.applications.resnet50.preprocess_input, 'ResNet50')
results['ResNet50'] = acc_resnet

acc_eff, hist_eff = train_transfer_model(EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input, 'EfficientNetB0')
results['EfficientNetB0'] = acc_eff


--- Training VGG16 ---
       0/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

## 5. Compare Performance

In [ ]:
names = list(results.keys())
accuracies = list(results.values())

plt.figure(figsize=(8, 5))
plt.bar(names, accuracies, color=['blue', 'green', 'orange'])
plt.title('Transfer Learning Models Comparison (CIFAR-10 Subset)')
plt.ylabel('Test Accuracy')
plt.ylim(0, 1.0)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')
plt.show()